In [55]:
from io import StringIO
from IPython.core.magic import register_cell_magic
from magic_codec.grammar import parse_grammar_str, generate_python
from pegen.tokenizer import Tokenizer
from pegen.parser import memoize, memoize_left_rec, logger, Parser

from magic_codec.parser.peg import *
from tokenize import TokenInfo, generate_tokens

grammar: dict[str, any] = {}
parser: dict[str, any] = {}

@register_cell_magic
def peg(name, cell):
    parsed = parse_grammar_str(cell)
    print(parsed)
    global grammar
    grammar[name] = parsed

@register_cell_magic
def parse(args, cell):
    name, *rules = args.split(' ')
    class_name = grammar[name].metas['class']
    _parser = globals()[class_name](Tokenizer(generate_tokens(StringIO(cell).readline)))
    result = None
    if rules:
        assert len(rules) == 1, "Too many args"
        result = getattr(_parser, rules[0])()
    else:
        result = _parser.start()
    print(result)

def generate(name: str, print_code=False):
    global parser
    parser[name] = generate_python(grammar[name])
    if print_code:
        print(parser[name])
    exec(parser[name], globals=globals())

In [56]:
%%peg test
# Minimal patch grammar for declarative macros

@class MacroPegParser
@extends "peg.py"
@base PegParser
@subheader"""
from magic_codec.parser.peg import *
from tokenize import TokenInfo

class Replacement:
    if sys.version_info >= (3, 10):
        __match_args__ = ("name")

    name: str
    _fields = ("name",)
    _field_types = {'name': str}

    def __init__(self, name: str):
        self.name = name
    
    def __repr__(self):
        return f"{self.name}"

    __str__ = __repr__

class Fragment:
    if sys.version_info >= (3, 10):
        __match_args__ = ("data")

    data: list[TokenInfo | Replacement]
    _fields = ("data",)
    _field_types = {'data': list[TokenInfo | Replacement]}

    def __init__(self, *args):
        self.data = self.flatten(args)

    def __repr__(self):
        return ', '.join(repr(arg) for arg in self.data)

    __str__ = __repr__

    def flatten(self, nested_list):
        for item in nested_list:
            if item is None:
                continue
            if isinstance(item, list):
                yield from self.flatten(item)
            else:
                yield item

"""
@trailer''

ANY: ~ { t if (t:=self._tokenizer.getnext()).type != 0 else None }
braces: '(' | ')' | '[' | ']' | '{' | '}'

unparsed_atom:
    | '$' a=NAME { Replacement(a) }
    | a=(!braces !NEWLINE !'$' ANY) { Fragment(a) }

unparsed_balanced:
    | a='(' b=unparsed_balanced? c=')' d=unparsed_balanced? { Fragment(a, b, c, d) } 
    | a='[' b=unparsed_balanced? c=']' d=unparsed_balanced? { Fragment(a, b, c, d) }
    | a='{' b=unparsed_balanced? c='}' d=unparsed_balanced? { Fragment(a, b, c, d) }
    | a=INDENT NEWLINE b=unparsed_balanced c=DEDENT { Fragment(a, b, c) }
    | a=unparsed_atom b=unparsed_balanced? { Fragment(a, b) }

unparsed_line:
    | !INDENT !DEDENT a=unparsed_balanced b=NEWLINE { Fragment(a, b) }

unparsed_block:
    | a=INDENT ~ b=unparsed_block+ c=DEDENT { Fragment(a, b, c) }
    | a=unparsed_line { Fragment(a) }

suite_fragment:
    | NEWLINE INDENT ~ a=unparsed_block+ DEDENT { Fragment(a) }
    | a=unparsed_line { Fragment(a) }

action[list]: 
    | ':' ~ a=suite_fragment { a }
    | !':' "{" ~ a=unparsed_balanced "}" { Fragment(a) }

ANY: ~
braces: '(' | ')' | '[' | ']' | '{' | '}'
unparsed_atom: '$' NAME | (!braces !NEWLINE !'$' ANY)
unparsed_balanced:
    | '(' unparsed_balanced? ')' unparsed_balanced?
    | '[' unparsed_balanced? ']' unparsed_balanced?
    | '{' unparsed_balanced? '}' unparsed_balanced?
    | INDENT NEWLINE unparsed_balanced DEDENT
    | unparsed_atom unparsed_balanced?
unparsed_line: !INDENT !DEDENT unparsed_balanced NEWLINE
unparsed_block: INDENT ~ unparsed_block+ DEDENT | unparsed_line
suite_fragment: NEWLINE INDENT ~ unparsed_block+ DEDENT | unparsed_line
action: ':' ~ suite_fragment | !':' "{" ~ unparsed_balanced "}"


In [57]:
generate("test")

In [60]:
%%parse test action
:
    if True:
        print("y")
    else:
        print("x")

TokenInfo(type=1 (NAME), string='if', start=(2, 4), end=(2, 6), line='    if True:\n'), TokenInfo(type=1 (NAME), string='True', start=(2, 7), end=(2, 11), line='    if True:\n'), TokenInfo(type=55 (OP), string=':', start=(2, 11), end=(2, 12), line='    if True:\n'), TokenInfo(type=4 (NEWLINE), string='\n', start=(2, 12), end=(2, 13), line='    if True:\n'), TokenInfo(type=5 (INDENT), string='        ', start=(3, 0), end=(3, 8), line='        print("y")\n'), TokenInfo(type=1 (NAME), string='print', start=(3, 8), end=(3, 13), line='        print("y")\n'), TokenInfo(type=55 (OP), string='(', start=(3, 13), end=(3, 14), line='        print("y")\n'), TokenInfo(type=3 (STRING), string='"y"', start=(3, 14), end=(3, 17), line='        print("y")\n'), TokenInfo(type=55 (OP), string=')', start=(3, 17), end=(3, 18), line='        print("y")\n'), TokenInfo(type=4 (NEWLINE), string='\n', start=(3, 18), end=(3, 19), line='        print("y")\n'), TokenInfo(type=6 (DEDENT), string='', start=(4, 4), en

In [ ]:
%%parse test start
foo: 
 | STRING: print("x")

None
